# Titanic Survival Prediction

End-to-end binary classification using Logistic Regression on the Kaggle Titanic dataset.

**Pipeline:**  
`Load Data` → `Explore Features` → `Drop Columns` → `Encode Categoricals` → `Impute Missing` → `Engineer Features` → `Train Model` → `Predict` → `Evaluate` → `Submit`

## 1. Imports

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings("ignore")

## 2. Load Data

In [2]:
df_train = pd.read_csv("train.csv")
df_test  = pd.read_csv("test.csv")
df_gen   = pd.read_csv("gender_submission.csv")  # submission template

print(f"Train shape : {df_train.shape}")
print(f"Test shape  : {df_test.shape}")
df_train.head()

Train shape : (891, 12)
Test shape  : (418, 11)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## 3. Exploratory Data Analysis

In [3]:
# Inspect object-type columns
df_train.select_dtypes(include=object)

,Name,Sex,Ticket,Cabin,Embarked
0,"Braund, Mr. Owen Harris",male,A/5 21171,NaN,S
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,PC 17599,C85,C
2,"Heikkinen, Miss. Laina",female,STON/O2. 3101282,NaN,S
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,113803,C123,S
4,"Allen, Mr. William Henry",male,373450,NaN,S
...,...,...,...,...,...
886,"Montvila, Rev. Juozas",male,211536,NaN,S
887,"Graham, Miss. Margaret Edith",female,112053,B42,S
888,"Johnston, Miss. Catherine Helen ""Carrie""",female,W./C. 6607,NaN,S
889,"Behr, Mr. Karl Howell",male,111369,C148,C


In [4]:
# Ticket has 681 unique values — too noisy to encode, will be dropped
print("Unique tickets:", df_train["Ticket"].nunique())

Unique tickets: 681


In [5]:
# Cabin: mostly missing; passengers with no cabin record skewed toward not surviving
print(df_train["Cabin"].value_counts().head(20))
df_train[(df_train["Cabin"].isnull()) & (df_train["Survived"] == 0)][["Survived", "Cabin"]]

Cabin
G6             4
C23 C25 C27    4
B96 B98        4
F2             3
D              3
E101           3
C22 C26        3
F33            3
C83            2
C123           2
B28            2
D26            2
B58 B60        2
E33            2
D33            2
C52            2
F G73          2
B77            2
C93            2
B5             2
Name: count, dtype: int64


,Survived,Cabin
0,0,NaN
4,0,NaN
5,0,NaN
7,0,NaN
12,0,NaN
...,...,...
884,0,NaN
885,0,NaN
886,0,NaN
888,0,NaN


In [6]:
# Target distribution (~38% survived)
df_train["Survived"].value_counts()

Survived
0    549
1    342
Name: count, dtype: int64

## 4. Preprocess — Training Set

In [7]:
# Drop high-cardinality / low-signal columns
df_train.drop(columns=["Name", "Ticket", "Cabin"], inplace=True)

In [8]:
# One-hot encode Sex and Embarked (drop_first avoids multicollinearity)
df_train = pd.get_dummies(df_train, columns=["Sex", "Embarked"], drop_first=True)
df_train.head()

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare,Sex_male,Embarked_Q,Embarked_S
0,1,0,3,22.0,1,0,7.2500,True,False,True
1,2,1,1,38.0,1,0,71.2833,False,False,False
2,3,1,3,26.0,0,0,7.9250,False,False,True
3,4,1,1,35.0,1,0,53.1000,False,False,True
4,5,0,3,35.0,0,0,8.0500,True,False,True


In [9]:
# Check missing values
df_train.isna().sum()

PassengerId      0
Survived         0
Pclass           0
Age            177
SibSp            0
Parch            0
Fare             0
Sex_male         0
Embarked_Q       0
Embarked_S       0
dtype: int64

In [10]:
# Age distribution before imputation
df_train["Age"].value_counts()

Age
24.00    30
22.00    27
18.00    26
28.00    25
30.00    25
         ..
24.50     1
0.67      1
0.42      1
34.50     1
74.00     1
Name: count, Length: 88, dtype: int64

In [11]:
# Age vs survival — no clean monotonic pattern; median imputation is safe
df_train.groupby("Age")["Survived"].agg(["min", "max", "count"])

,min,max,count
Age,,,
0.42,1,1,1
0.67,1,1,1
0.75,1,1,2
0.83,1,1,2
0.92,1,1,1
...,...,...,...
70.00,0,0,2
70.50,0,0,1
71.00,0,0,2


In [12]:
# Impute Age with median
df_train["Age"] = df_train["Age"].fillna(df_train["Age"].median())

In [13]:
# Feature engineering: invert Age so higher value = younger passenger
# (100 - Age) keeps scale intuitive while flipping the direction)
df_train["Age"] = 100 - df_train["Age"]
df_train.head()

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare,Sex_male,Embarked_Q,Embarked_S
0,1,0,3,78.0,1,0,7.2500,True,False,True
1,2,1,1,62.0,1,0,71.2833,False,False,False
2,3,1,3,74.0,0,0,7.9250,False,False,True
3,4,1,1,65.0,1,0,53.1000,False,False,True
4,5,0,3,65.0,0,0,8.0500,True,False,True


In [14]:
# Confirm no missing values remain
df_train.isna().sum()

PassengerId    0
Survived       0
Pclass         0
Age            0
SibSp          0
Parch          0
Fare           0
Sex_male       0
Embarked_Q     0
Embarked_S     0
dtype: int64

## 5. Model Training

In [15]:
# Define features and target
X_train = df_train.drop(columns=["Survived", "PassengerId"])
y_train = df_train["Survived"]

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

X_train shape: (891, 8)
y_train shape: (891,)


In [16]:
# Fit Logistic Regression
lr = LogisticRegression()
lr.fit(X_train, y_train)

LogisticRegression()

## 6. Preprocess — Test Set

In [17]:
# Mirror the same cleaning steps applied to training data
df_test.drop(columns=["PassengerId", "Name", "Ticket", "Cabin"], inplace=True)
df_test = pd.get_dummies(df_test, columns=["Sex", "Embarked"], drop_first=True)
df_test.head()

,Pclass,Age,SibSp,Parch,Fare,Sex_male,Embarked_Q,Embarked_S
0,3,34.5,0,0,7.8292,True,True,False
1,3,47.0,1,0,7.0000,False,False,True
2,2,62.0,0,0,9.6875,True,True,False
3,3,27.0,0,0,8.6625,True,False,True
4,3,22.0,1,1,12.2875,False,False,True


In [18]:
# Check missing values in test set
df_test.isna().sum()

Pclass         0
Age           86
SibSp          0
Parch          0
Fare           1
Sex_male       0
Embarked_Q     0
Embarked_S     0
dtype: int64

In [19]:
# Impute Age and the single missing Fare
df_test["Age"]  = df_test["Age"].fillna(df_test["Age"].median())
df_test["Fare"] = df_test["Fare"].fillna(df_test["Fare"].median())

In [20]:
# Apply same Age inversion as training set
df_test["Age"] = 100 - df_test["Age"]
df_test.head()

,Pclass,Age,SibSp,Parch,Fare,Sex_male,Embarked_Q,Embarked_S
0,3,65.5,0,0,7.8292,True,True,False
1,3,53.0,1,0,7.0000,False,False,True
2,2,38.0,0,0,9.6875,True,True,False
3,3,73.0,0,0,8.6625,True,False,True
4,3,78.0,1,1,12.2875,False,False,True


## 7. Predict & Validate

In [21]:
# Generate predictions
y_pred = lr.predict(df_test)
print("Predictions:", y_pred)

Predictions: [0 0 0 0 1 0 1 0 1 0 0 0 1 0 1 1 0 0 1 1 0 0 1 1 1 0 1 0 0 0 0 0 0 1 0 0 1
 1 0 0 0 0 0 1 1 0 0 0 1 1 0 0 1 1 0 0 0 0 0 1 0 0 0 1 1 1 1 0 1 1 1 0 1 1
 1 1 0 1 0 1 0 0 0 0 0 0 1 1 1 0 1 0 1 0 1 0 1 0 1 0 1 0 0 0 1 0 0 0 0 0 0
 1 1 1 1 0 0 1 1 1 1 0 1 0 0 1 0 1 0 0 0 0 1 0 0 0 0 0 1 0 0 1 0 0 0 0 1 0
 0 0 1 0 0 1 0 0 1 1 0 1 1 0 1 0 0 1 0 0 1 1 0 0 0 0 0 1 1 0 1 1 0 0 1 0 1
 0 1 0 0 0 0 0 0 0 0 0 1 1 0 1 1 0 0 1 0 1 1 0 1 0 0 0 0 0 0 0 1 0 1 0 1 0
 1 0 1 1 0 1 0 0 0 1 0 0 0 0 0 0 1 1 1 1 0 0 0 0 1 0 1 1 1 0 1 0 0 0 0 0 1
 0 0 0 1 1 0 0 0 0 1 0 0 0 1 1 0 1 0 0 0 0 1 0 1 1 1 0 0 0 0 0 0 1 0 0 0 0
 1 0 0 0 0 0 0 0 1 1 0 0 0 0 0 0 0 1 1 1 0 0 0 0 0 0 0 0 1 0 1 0 0 0 1 0 0
 1 0 0 0 0 0 0 0 0 0 1 0 1 0 1 0 1 1 0 0 0 1 0 1 0 0 1 0 1 1 0 1 1 0 1 1 0
 0 1 0 0 1 1 0 0 0 0 0 0 1 1 0 1 0 0 0 0 1 1 0 0 0 1 0 1 0 0 1 0 1 1 0 0 0
 0 1 1 1 1 1 0 1 0 0 0]


In [22]:
# Sanity check: prediction distribution should resemble training distribution
df_gen["Survived"] = y_pred

pred_dist  = df_gen["Survived"].value_counts() / len(df_gen["Survived"]) * 100
train_dist = df_train["Survived"].value_counts() / len(df_train["Survived"]) * 100

print("Test prediction distribution (%):\n", pred_dist)
print("\nTrain label distribution (%):\n", train_dist)

Test prediction distribution (%):
 Survived
0    62.440191
1    37.559809
Name: count, dtype: float64

Train label distribution (%):
 Survived
0    61.616162
1    38.383838
Name: count, dtype: float64


## 8. Export Submission

In [26]:
df_gen.to_csv("submission.csv", index=False)
print("submission.csv saved — ready to upload to Kaggle.")

submission.csv saved — ready to upload to Kaggle.
